# Đề tài 13 – Đánh giá rủi ro tín dụng cho Mua trước Trả sau (BNPL)
## Notebook 01 – Khám phá dữ liệu (EDA)

**Dữ liệu:** Home Credit – Credit Risk Model Stability (Kaggle, 2024) – dữ liệu thực tế.
**Nguồn:** https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability

Mục tiêu notebook: kiểm chứng kích thước, khoảng thời gian, tỷ lệ vỡ nợ, mức độ thiếu dữ liệu và độ ổn định theo thời gian.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl
import matplotlib.pyplot as plt, seaborn as sns
import utils
from utils import META

DATA_DIR, OUT_DIR = utils.get_data_dir(), utils.get_out_dir()
print("Du lieu:", DATA_DIR); print("Ket qua:", OUT_DIR)

## 1. Bảng gốc: kích thước, khoảng thời gian, tỷ lệ vỡ nợ

In [ ]:
base = utils.collect(utils.load_base())
print("So ho so:", base.height)
print("Khoang ngay quyet dinh:", base["date_decision"].min(), "->", base["date_decision"].max())
print("So tuan:", base["WEEK_NUM"].n_unique())
print("Ty le vo no (target=1): %.2f%%" % (100 * base["target"].mean()))
base.head()

> **Ghi vào báo cáo:** số hồ sơ, khoảng ngày (chứng minh yêu cầu *timeliness 7 năm*), tỷ lệ vỡ nợ (mức độ mất cân bằng).

In [ ]:
wk = (base.group_by("WEEK_NUM").agg(pl.len().alias("n"), pl.col("target").mean().alias("bad_rate"))
          .sort("WEEK_NUM").to_pandas())
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].bar(wk["WEEK_NUM"], wk["n"]); ax[0].set_title("So ho so theo tuan"); ax[0].set_xlabel("WEEK_NUM")
ax[1].plot(wk["WEEK_NUM"], 100 * wk["bad_rate"]); ax[1].set_title("Ty le vo no theo tuan (%)"); ax[1].set_xlabel("WEEK_NUM")
plt.tight_layout(); plt.show()

## 2. Tổng quan các bảng sử dụng

In [ ]:
TABLES = ["static_0", "static_cb_0", "applprev_1", "person_1", "credit_bureau_a_1"]
rows = []
for t in TABLES:
    try:
        lf = utils.scan_table(t)
        sch = lf.collect_schema()
        n = lf.select(pl.len()).collect().item()
        n_case = lf.select(pl.col("case_id").n_unique()).collect().item()
        rows.append({"bang": t, "so_file": len(utils.table_files(t)), "so_dong": n, "so_ho_so": n_case,
                     "dong/ho_so": round(n / n_case, 2), "so_cot": len(sch)})
    except FileNotFoundError as e:
        print(e)
pd.DataFrame(rows)

## 3. Giá trị thiếu và quan hệ với biến mục tiêu (bảng `static_0`)

In [ ]:
SAMPLE = 0.2   # lay mau 20% ho so de EDA nhanh; dat None de dung toan bo
b = utils.load_base(SAMPLE)
st = utils.collect(b.join(utils.depth0_features("static_0", b), on="case_id", how="left"))
st = st.join(utils.collect(utils.depth0_features("static_cb_0", b)), on="case_id", how="left")
print(st.shape)

miss = (st.null_count() / st.height).transpose(include_header=True, header_name="cot", column_names=["ty_le_thieu"]).to_pandas()
miss = miss.sort_values("ty_le_thieu", ascending=False)
miss["ty_le_thieu"].plot.hist(bins=20, title="Phan phoi ty le thieu cua cac cot", figsize=(6, 3)); plt.show()
miss.head(15)

In [ ]:
# Information Value (IV) tren mau: xep hang suc phan biet cua tung bien
pdf = utils.prune_features(st).to_pandas()
feats = [c for c in pdf.columns if c not in META]
woe = utils.WoEEncoder().fit(pdf[feats], pdf["target"])
iv = woe.iv_
print("IV > 0.5 can nghi ngo ro ri du lieu:", list(iv[iv > 0.5].index))
iv.head(20).iloc[::-1].plot.barh(figsize=(7, 6), title="Top 20 bien theo IV"); plt.show()

In [ ]:
# Ty le vo no theo bin cua 6 bien so manh nhat
top = [c for c in iv.index if pd.api.types.is_numeric_dtype(pdf[c])][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), top):
    q = pd.qcut(pdf[c], 10, duplicates="drop")
    (100 * pdf.groupby(q, observed=True)["target"].mean()).plot.bar(ax=ax)
    ax.set_title(c); ax.set_ylabel("% vo no"); ax.tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

## 4. Kiểm tra rò rỉ theo thời gian
Cột ngày đã được đổi thành *số ngày so với ngày quyết định*. Giá trị dương = sau ngày quyết định → cần xem ý nghĩa cột (ví dụ ngày đến hạn dự kiến là hợp lệ, ngày trả nợ thực tế thì không).

In [ ]:
date_cols = [c for c in pdf.columns if c.endswith("D") and c not in META]
fut = pd.Series({c: float((pdf[c] > 0).mean()) for c in date_cols}).sort_values(ascending=False)
fut[fut > 0].to_frame("ty_le_gia_tri_sau_ngay_quyet_dinh")

## 5. Nhận xét (tự viết)
- Mức độ mất cân bằng: ...
- Xu hướng tỷ lệ vỡ nợ theo thời gian: ...
- Các biến có sức phân biệt cao và ý nghĩa nghiệp vụ: ...
- Biến nghi ngờ rò rỉ cần loại: ...